Notebook to create some of the visualization.

In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from confidence.model.single_pass import SinglePassConfidence
#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "tu_berlin"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}


architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:

from get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
from src.utils.eval.vis import vis_dataset

x = next(iter(test_loader_transformed))[0]

batch_size = next(iter(train_loader))[0].shape[0]


vis_dataset(train_loader, val_loader, test_loader_transformed)


model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparison_unsupervised")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
from model.get_model import get_network

from model.train import train_and_get_model

model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
from src.utils.eval.main_model import evaluate_base_model

#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
from transformation import get_transformation_sequence_images
from src.utils.transforms.scale import Scale, DirectedScale, ScaleAllSame

is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

from src.utils.replacer import replace_rotation_transforms_2vec,replace_scaling_transforms,replace_rotation_transforms


from src.utils.transforms.apply import grid_resample

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
print(transform_seq.transformations)

In [ ]:
from model.basic_networks import make_deterministic

make_deterministic(model)



In [ ]:
from get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from model.basic_networks import FlexibleResNet

detectors = ["knn","per_class_knn", "knn_mixed","per_class_knn_mixed","knn_mixed_faiss","knn_itf","vim","react","dice","ash","she","laplace_mi","laplace_energy","laplace_weighted","trust_score","openmax","mahalanobis","rmd","class_prototype","react_all","energy","per_class_prototype","prototype_multi", "laplace_entropy","adjusted_entropy"]
#dont do kde to expensive, and not better.
#global protoype makes no sense over class based protoype
# lof fit is to slow, gmm is even slower or fails due to size
# gram runs out of main memory,

detectors_parameterless = ["energy_ts","entropy","kl_matching","laplace_entropy_gridsearch"]
detectors = detectors + detectors_parameterless
if not isinstance(model,FlexibleResNet):
    #remove ash and react_all as they only work with resnets
    detectors.remove("ash")
    detectors.remove("react_all")
    if "react" not in detectors:
        detectors.append("react")
    detectors.append("ash_last")



In [ ]:

import os
import json
import math
import torch
from typing import Tuple, Optional, Any, Dict


# Example call (update unpacking to 6 values)
base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)
from hyper_param.ood.base_prepare import find_best_detector_and_instantiate #TODO only hyper_param.ood.base_prepare works not ood.base_prepare directly

best_detector, best_problem, best_score, second_choice, second_problem, second_score = find_best_detector_and_instantiate(
    base_results_dir=base_results_dir,
    detectors=detectors,
    model=model,
    train_cache=train_cache,
    transform_seq_arg=transform_seq,
    dataset_info=dataset_info,
    architecture=architecture,
    device=device,
    val_id_loader=val_loader_transformed,
    val_ood_loader=val_loader_transformed,
)

In [ ]:
#here we do the following calculate the accuracy on the test set, then calculate on the transformed test set. We also store the confidence valeus
#then we do the optimiztation to find which ones we can transform to get better accuracy.
#we then draw the curve over the threshold when we switch from doing no optimization to doing optimization, this should drop the accuracy on the test set but increase on the transformed test set.



In [ ]:
transform_seq_spatial = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
).to(device)
from src.utils.transforms.scale import Scale, DirectedScale, ScaleAllSame

is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]




from src.utils.transforms.apply import grid_resample

transform_seq2 = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

if dataset == "modelnet10":
    transform_seq2 = replace_rotation_transforms_2vec(transform_seq2)
else:
    transform_seq2 = replace_rotation_transforms(transform_seq)
if dataset=="coil100":
    transform_seq2 = replace_scaling_transforms(transform_seq2, replace_scale=False)
transform_seq_spatial = transform_seq2




In [ ]:
train_loader_augmented = dataset_dict.get('train_loader_augmented', None)

In [ ]:
from model.get_model import get_network_layer

layer,layer_io = get_network_layer(dataset_info, architecture, 1, num_classes=None, num_rotations=8)

In [ ]:
from embedding_cache import LayerEmbeddingCache
cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

cache_train = LayerEmbeddingCache(model, train_loader_no_shuffle,
                                  cache_dir=os.path.join(embedding_cache_path, cache_name_train))

dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)
embeddings_t, final_t, classes_t = cache_train.__call__(layer, capture_modes=layer_io, flatten=True)


In [ ]:
from src.utils.transformation_problem import TransformationProblem
from confidence.direct.targeted import CrossEntropyConfidence
from confidence.model.single_pass import SinglePassConfidence
from confidence.unsupervised.classic.prototype import ClassPrototypeConfidence
from confidence.control.split import TrueLabelSplitConfidence

nn_pytorch_pretrained = ClassPrototypeConfidence(metric="cosine")
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.to(device)

conf_split_pretrained = TrueLabelSplitConfidence(nn_pytorch_pretrained, CrossEntropyConfidence(), mult=False,a=10,b=1)


conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_guided = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq_spatial,consolidate_method="consolidate_simple")
model.eval().to(device)

In [ ]:
transform_seq_spatial.transformations

In [ ]:
from dataset.geometric_wrapper import TransformParamDataset
import importlib
import tqdm
import pickle
import search
import search.random_search
train_loader_transformed_no_shuffle = dataset_dict['train_loader_transformed_no_shuffle']





# Define cache path
regression_cache_path = os.path.join(
    embedding_cache_path,
    f"regression_params_{dataset}_{architecture}_{transform_name}_test_set_60.pkl"
)

if os.path.exists(regression_cache_path):
    print(f"Loading cached regression parameters from {regression_cache_path}")
    with open(regression_cache_path, 'rb') as f:
        cache_data = pickle.load(f)
        regression_target = cache_data['regression_target']
        test_acc_sim = cache_data['test_acc_sim']
else:
    print("Computing regression parameters (this may take a while)...")

    di = search.random_search.RSLR(initial_samples=46, local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})
    if dataset == "tu_berlin":
        di = search.random_search.RSLR(initial_samples=60,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})

    model.eval().cuda()

    regression_target = []
    test_acc_sim = 0.0
    counter = 0
    problem_guided.max_batch_size =dataset_info.batch_size_search
    bar = tqdm.tqdm(test_loader_transformed)
    first=True
    print()
    try:
        for data, target in bar:
            data, target = data.cuda(), target.cuda()
            res = di.optimize(problem_guided, data.cuda(), y=target.cuda())
            zw = transform_seq_spatial.normalize(res[0])
            regression_target.append(zw.detach().cpu())

            # Apply the transformation and get predictions
            x_transformed2 = problem_guided.transform(data.cuda(), zw)
            logits = model(x_transformed2)
            output = logits.argmax(dim=-1)
            test_acc_sim += output.eq(target).sum().item()
            counter += output.shape[0]
            #update bar with intermediate accuracy
            if first and is_image_data:
                first=False
                #plot
                import matplotlib.pyplot as plt
                grid_orig = torchvision.utils.make_grid(data.cpu(), nrow=8, normalize=True, scale_each=True)
                grid_transformed = torchvision.utils.make_grid(x_transformed2.cpu(), nrow=8, normalize=True, scale_each=True)
                plt.figure(figsize=(12, 6))
                plt.subplot(1, 2, 1)
                plt.title("Original Images")
                plt.imshow(grid_orig.permute(1, 2, 0))
                plt.axis('off')
                plt.subplot(1, 2, 2)
                plt.imshow(grid_transformed.permute(1, 2, 0))
                plt.title("Transformed Images")
                plt.show()
                print()
            #flush bar
            bar.refresh()
            bar.set_description(f"Intermediate transformed accuracy: {test_acc_sim / counter:.4f}")

    finally:
        bar.close()

    test_acc_sim /= counter
    regression_target = torch.cat(regression_target, dim=0)

    # Save to cache
    os.makedirs(os.path.dirname(regression_cache_path), exist_ok=True)
    with open(regression_cache_path, 'wb') as f:
        pickle.dump({
            'regression_target': regression_target,
            'test_acc_sim': test_acc_sim
        }, f)
    print(f"Cached regression parameters to {regression_cache_path}")



regression_dataset = TransformParamDataset(test_loader_transformed.dataset, regression_target)
regression_dataloader = torch.utils.data.DataLoader(
    regression_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    persistent_workers=True
)

print(f'With class knowledge we can define a canonical representation of the data: {test_acc_sim}.')



In [ ]:
x = next(iter(regression_dataloader))[0]

In [ ]:
test_loader_transformed.dataset[0]


In [ ]:
regression_dataset[0]

In [ ]:
#test the regression true dataset
if is_image_data:
    x,p,y =next(iter(regression_dataloader))
    T = transform_seq_spatial(p)
    #T = torch.linalg.inv(T)
    x_untransformed = transform_seq_spatial.application_method(x, T)

    plt.figure(figsize=(12, 6))
    grid_orig = torchvision.utils.make_grid(x.cpu(), nrow=8, normalize=True, scale_each=True)
    grid_untransformed = torchvision.utils.make_grid(x_untransformed.cpu(), nrow=8, normalize=True, scale_each=True)
    plt.subplot(1, 2, 1)
    plt.title("Transformed Images")
    plt.imshow(grid_orig.permute(1, 2, 0))
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(grid_untransformed.permute(1, 2, 0))
    plt.title("Untransformed Images")
    plt.show()






In [ ]:
from search.random_search import RSLR

optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})

if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})

In [ ]:
from src.utils.eval.ood_performance import ITSWRAPPER
from its.search import InverseTransformationSearch

its_opt = None
extend=0
if dataset == "tu_berlin":
    its_opt = ITSWRAPPER(InverseTransformationSearch(model, None, None, n_hypotheses=1, n_samples=12,extend=extend,
                                                             gaussian_filter_channel_wise=False))
elif dataset == "modelnet10":
    its_opt = ITSWRAPPER(InverseTransformationSearch(model, None, None, n_hypotheses=4, n_samples=6,extend=extend,
                                                             gaussian_filter_channel_wise=True))
elif dataset == "coil100":
    its_opt = ITSWRAPPER(InverseTransformationSearch(model, None, None, n_hypotheses=1, n_samples=10,extend=extend,
                                                             gaussian_filter_channel_wise=True))
else:
    its_opt = ITSWRAPPER(InverseTransformationSearch(model, None, None, n_hypotheses=2, n_samples=6,extend=extend,
                                                             gaussian_filter_channel_wise=False))


In [ ]:
dataset

In [ ]:

from hyper_param.ood.base_prepare import create_ood_problem, get_default_ood_params
import os
import json
import math
import torch
from typing import Tuple, Optional, Any, Dict


# Example call (update unpacking to 6 values)
base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)

best_detector, best_problem, best_score, second_choice, second_problem, second_score = find_best_detector_and_instantiate(
    base_results_dir=base_results_dir,
    detectors=detectors,
    model=model,
    train_cache=train_cache,
    transform_seq_arg=transform_seq,
    dataset_info=dataset_info,
    architecture=architecture,
    device=device,
    val_id_loader=val_loader_transformed,
    val_ood_loader=val_loader_transformed,
)

In [ ]:
best_problem.max_batch_size=dataset_info.batch_size_search

In [ ]:

import torch

# Get a single batch (data, target) from the test loader
data, target = next(iter(test_loader_transformed))
data, target = data.cuda(), target.cuda()
data = data[8:9]  # Use only first 5 samples for testing
target = target[8:9]

# Number of repeated optimizations
num_repeats = 5
results = []
results_images = []

for i in range(num_repeats):
    res = optimizer.optimize(best_problem, data, y=target)
    zw = res[0]
    results.append(zw.detach().cpu())
    results_images.append(best_problem.transform(data, zw).detach().cpu())

# Compare the results
for idx, params in enumerate(results):
    print(f"Run {idx+1}: {params}")

if is_image_data:
    fig, axes = plt.subplots(1, num_repeats + 1, figsize=(3 * (num_repeats + 1), 3))
    # Show original image
    axes[0].imshow(data[0].cpu().permute(1, 2, 0).squeeze(), cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")
    # Apply each transformation and show result
    for i, params in enumerate(results_images):
        x_transformed = results_images[i]
        axes[i + 1].imshow(x_transformed[0].cpu().permute(1, 2, 0).squeeze(), cmap="gray")
        axes[i + 1].set_title(f"Run {i+1}")
        axes[i + 1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
dataset

In [ ]:

from model.train import train_or_load_energy_model
from src.utils.sampling import BatchNegativeSampler
from src.utils.augments import small_affine_augment_2d, ComposeAugmentations, random_blur_or_sharpen
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from src.utils.transformation_problem import TransformationProblem

architecture_quarter = architecture+"_quarter"

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None

def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y




optimizer_240 = RSLR(initial_samples=195, local_runs=5, local_max_steps=4,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer_240 = RSLR(initial_samples=240,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})
if dataset == "modelnet10":
    optimizer_240 = RSLR(initial_samples=180-27, local_runs=3, local_max_steps=4, local_opt_kwargs={"lr": 0.1})




optimizer_240_include_init = RSLR(initial_samples=195, local_runs=5, local_max_steps=4,local_opt_kwargs={"lr":0.1},include_zero_always=True)
if dataset == "tu_berlin":
    optimizer_240_include_init = RSLR(initial_samples=240,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1},include_zero_always=True)
if dataset == "modelnet10":
    optimizer_240_include_init = RSLR(initial_samples=180-27, local_runs=3, local_max_steps=4, local_opt_kwargs={"lr": 0.1},include_zero_always=True)





energy_model2_quarter = get_network(dataset_info,architecture_quarter, num_classes=1).to(device)
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    = transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_quarter = train_or_load_energy_model(
    energy_model2_quarter, model_dir_path, f"{modelname}_energy2_quarter", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_quarter.to(device).eval()
problem_energy_quarter = TransformationProblem(energy_conf_quarter, transform_seq,                                              consolidate_method="consolidate_simple")

def savepath_later_layer(label: str) -> str:
    model_dir_path = os.path.join(current_path, "experiment_files", "models")
    embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
    # Add results dir and helper for save paths
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

name = "learned_energy_confidence_quarter_transformed_240"
name_un = "learned_energy_confidence_quarter_untransformed_240"
name_un_included = "learned_energy_confidence_quarter_untransformed_240_inc"
if dataset == "modelnet10":
    name = "learned_energy_confidence_quarter_transformed_180"
    name_un = "learned_energy_confidence_quarter_untransformed_180"
    name_un_included = "learned_energy_confidence_quarter_untransformed_180_inc"


res_large_transformed = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath_later_layer(name), show_progress=True,
    repeats=5)

res_large_real_data = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer(name_un),
    repeats=5)

res_large_real_data_included = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240_include_init, problem=problem_energy_quarter,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer(name_un_included),
    repeats=5)

print(res_large_real_data)
print(res_large_real_data_included)

del energy_model2_quarter
del energy_conf_quarter








In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from typing import Optional, Tuple



In [ ]:
from typing import List

def visualize_single_detector_transformations(
    model,
    optimizer,
    problem,
    test_loader,
    dataset_info,
    n_rows: int = 2,
    n_cols: int = 4,
    device: str = "cuda",
    save_prefix: Optional[str] = None,
    second_loader: Optional[torch.utils.data.DataLoader] = None,
    second_indices: Optional[list] = None
):
    """
    Visualize optimized transformations from a single detector/problem.
    Green border = correct classification, Red border = incorrect classification.
    Supports images, point clouds, and stroke sequences.

    Allows replacing specific images in the original batch with images from a second loader.
    Images from second_loader are NOT optimized.
    """
    model.eval()
    n_samples = n_rows * n_cols

    # Get a batch of data from the main loader
    data, target = next(iter(test_loader))
    data, target = data.to(device), target.to(device)
    data = data[:n_samples].clone()
    target = target[:n_samples].clone()

    # Initialize a mask for which samples should be optimized
    optimize_mask = torch.ones(n_samples, dtype=torch.bool, device=device)

    # If replacements are requested, insert them and skip optimization for those
    if second_loader is not None and second_indices is not None:
        second_data, second_target = next(iter(second_loader))
        second_data, second_target = second_data.to(device), second_target.to(device)
        for i, idx in enumerate(second_indices):
            if idx < n_samples and i < len(second_data):
                data[idx] = second_data[i]
                target[idx] = second_target[i]
                optimize_mask[idx] = False  # Skip optimization for these

    # Determine data type
    data_shape = data.shape
    is_image = len(data_shape) == 4 and data_shape[1] in [1, 3]
    is_pointcloud = len(data_shape) == 3 and data_shape[-1] == 3
    is_stroke = len(data_shape) == 3 and data_shape[-1] == 4

    # Optimize only the subset of samples that are not from second_loader
    x_optimized = data.clone()
    if optimize_mask.any():
        res = optimizer.optimize(problem, data[optimize_mask], y=target[optimize_mask])
        params = res[0]
        if isinstance(optimizer, ITSWRAPPER):
            x_optimized[optimize_mask] = problem.transform_sequence.application_method(
                data[optimize_mask], params
            )
        else:
            x_optimized[optimize_mask] = problem.transform(data[optimize_mask], params)

    # Get predictions
    with torch.no_grad():
        pred_orig = model(data).argmax(dim=-1)
        pred_opt = model(x_optimized).argmax(dim=-1)

    # Visualization
    if is_image:
        _visualize_images_with_borders(
            data, x_optimized, target, pred_orig, pred_opt,
            n_rows, n_cols, save_prefix
        )
    elif is_pointcloud:
        _visualize_pointclouds_with_borders(
            data, x_optimized, target, pred_orig, pred_opt,
            n_rows, n_cols, save_prefix
        )
    elif is_stroke:
        _visualize_strokes_with_borders(
            data, x_optimized, target, pred_orig, pred_opt,
            n_rows, n_cols, save_prefix
        )
    else:
        print(f"Unsupported data shape: {data_shape}")
        return

    # Print statistics
    acc_orig = (pred_orig == target).float().mean().item()
    acc_opt = (pred_opt == target).float().mean().item()
    print(f"Accuracy - Original: {acc_orig:.3f}, Optimized: {acc_opt:.3f}")

    return {"accuracy_original": acc_orig, "accuracy_optimized": acc_opt}



def _visualize_images_with_borders(
    data, x_optimized, target, pred_orig, pred_opt,
    n_rows, n_cols, save_prefix
):
    """Helper to visualize images with colored borders."""
    # Clip to [0, 1]
    data = torch.clamp(data, 0, 1)
    x_optimized = torch.clamp(x_optimized, 0, 1)

    def add_border(img_tensor, is_correct):
        """Add colored border to image tensor (C, H, W)"""
        color = torch.tensor([0.0, 0.45, 0.70] if is_correct else [0.85, 0.35, 0.25], device=img_tensor.device)
        bordered = img_tensor.clone()

        # Handle grayscale
        if bordered.shape[0] == 1:
            bordered = bordered.repeat(3, 1, 1)

        # Add border
        border_width = 1
        for c in range(3):
            bordered[c, :border_width, :] = color[c]
            bordered[c, -border_width:, :] = color[c]
            bordered[c, :, :border_width] = color[c]
            bordered[c, :, -border_width:] = color[c]

        return bordered

    # Create bordered images
    n_samples = len(data)
    data_bordered = torch.stack([add_border(data[i], pred_orig[i] == target[i]) for i in range(n_samples)])
    opt_bordered = torch.stack([add_border(x_optimized[i], pred_opt[i] == target[i]) for i in range(n_samples)])

    # Create grids
    grid_orig = torchvision.utils.make_grid(data_bordered.cpu(), nrow=n_cols, padding=2)
    grid_opt = torchvision.utils.make_grid(opt_bordered.cpu(), nrow=n_cols, padding=2)

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(12, 6 * n_rows / n_cols), facecolor='white')
    axes[0].imshow(grid_orig.permute(1, 2, 0))
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(grid_opt.permute(1, 2, 0))
    axes[1].set_title('Optimized')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Save if requested
    if save_prefix:
        fig_orig, ax_orig = plt.subplots(1, 1, figsize=(6, 6 * n_rows / n_cols), facecolor='white')
        ax_orig.imshow(grid_orig.permute(1, 2, 0))
        ax_orig.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_original.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_orig)

        fig_opt, ax_opt = plt.subplots(1, 1, figsize=(6, 6 * n_rows / n_cols))
        ax_opt.imshow(grid_opt.permute(1, 2, 0))
        ax_opt.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_optimized.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_opt)

        print(f"Saved: {save_prefix}_original.pdf and {save_prefix}_optimized.pdf")


def _visualize_pointclouds_with_borders(
    data, x_untransformed, target, pred_orig, pred_untrans,
    n_rows, n_cols, save_prefix
):
    """Helper to visualize point clouds with colored borders."""
    n_samples = len(data)

    c_correct = (0.0, 0.45, 0.70)
    c_incorrect = (0.85, 0.35, 0.25)

    # Display figure with both columns
    fig = plt.figure(figsize=(6 * n_cols, 3 * n_rows), facecolor='white')
    for i in range(n_samples):
        row, col = i // n_cols, i % n_cols

        # Original/Transformed
        ax1 = fig.add_subplot(n_rows, 2 * n_cols, row * 2 * n_cols + col + 1, projection='3d')
        ax1.set_facecolor('white')
        pts = data[i].cpu().numpy()
        color = c_correct if pred_orig[i] == target[i] else c_incorrect

        ax1.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, c=color,rasterized=True)
        ax1.set_xticks([])
        ax1.set_yticks([])
        ax1.set_zticks([])

        # Untransformed
        ax2 = fig.add_subplot(n_rows, 2 * n_cols, row * 2 * n_cols + n_cols + col + 1, projection='3d')
        ax2.set_facecolor('white')
        pts_opt = x_untransformed[i].cpu().numpy()
        color_opt = c_correct if pred_untrans[i] == target[i] else c_incorrect
        ax2.scatter(pts_opt[:, 0], pts_opt[:, 1], pts_opt[:, 2], s=2, c=color_opt,rasterized=True)
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.set_zticks([])

    plt.tight_layout()
    plt.show()

    # Save separate files
    if save_prefix:
        # Original/Transformed only
        fig_orig = plt.figure(figsize=(3 * n_cols, 3 * n_rows), facecolor='white')
        for i in range(n_samples):
            row, col = i // n_cols, i % n_cols
            ax = fig_orig.add_subplot(n_rows, n_cols, i + 1, projection='3d',rasterized=True)
            ax.set_facecolor('white')
            pts = data[i].cpu().numpy()
            color = c_correct if pred_orig[i] == target[i] else c_incorrect
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, c=color)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_zticks([])
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_original.pdf", bbox_inches='tight', pad_inches=0,dpi=75)
        plt.close(fig_orig)

        # Untransformed only
        fig_untrans = plt.figure(figsize=(3 * n_cols, 3 * n_rows), facecolor='white')
        for i in range(n_samples):
            row, col = i // n_cols, i % n_cols
            ax = fig_untrans.add_subplot(n_rows, n_cols, i + 1, projection='3d',rasterized=True)
            ax.set_facecolor('white')
            pts_opt = x_untransformed[i].cpu().numpy()
            color_opt = c_correct if pred_untrans[i] == target[i] else c_incorrect
            ax.scatter(pts_opt[:, 0], pts_opt[:, 1], pts_opt[:, 2], s=2, c=color_opt)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_zticks([])

        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_optimized.pdf", bbox_inches='tight', pad_inches=0,dpi=75)
        plt.close(fig_untrans)

        print(f"Saved: {save_prefix}_original.pdf and {save_prefix}_optimized.pdf")

def _visualize_strokes_with_borders(
    data, x_untransformed, target, pred_orig, pred_untrans,
    n_rows, n_cols, save_prefix
):
    """Helper to visualize stroke sequences with colored borders."""
    n_samples = len(data)

    c_correct = (0.0, 0.45, 0.70)
    c_incorrect = (0.85, 0.35, 0.25)

    # Display figure with both columns
    fig, axes = plt.subplots(n_rows, 2 * n_cols, figsize=(3 * 2 * n_cols, 3 * n_rows))
    if isinstance(axes, np.ndarray):
            axes = axes.reshape(n_rows, 2*n_cols)
    else:
            axes = np.array([[axes]])

    for i in range(n_samples):
        row, col = i // n_cols, i % n_cols

        # Original/Transformed
        ax1 = axes[row, col]
        ax1.set_facecolor('white')
        color = c_correct if pred_orig[i] == target[i] else c_incorrect
        _plot_stroke_on_ax(data[i].cpu(), ax1, color=color)

        # Untransformed
        ax2 = axes[row, n_cols + col]
        color_opt = c_correct if pred_untrans[i] == target[i] else c_incorrect
        _plot_stroke_on_ax(x_untransformed[i].cpu(), ax2, color=color_opt)

    # Hide extra subplots
    for i in range(n_samples, n_rows * n_cols):
        row, col = i // n_cols, i % n_cols
        axes[row, col].axis('off')
        axes[row, n_cols + col].axis('off')

    plt.tight_layout()
    plt.show()

    # Save separate files
    if save_prefix:
        # Original/Transformed only
        fig_orig, axes_orig = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
        if isinstance(axes_orig, np.ndarray):
            axes_orig = axes_orig.reshape(n_rows, n_cols)
        else:
            axes_orig = np.array([[axes_orig]])
        for i in range(n_samples):
            row, col = i // n_cols, i % n_cols
            ax = axes_orig[row, col]
            ax1.set_facecolor('white')
            color = c_correct if pred_orig[i] == target[i] else c_incorrect
            _plot_stroke_on_ax(data[i].cpu(), ax, color=color)
        for i in range(n_samples, n_rows * n_cols):
            row, col = i // n_cols, i % n_cols
            axes_orig[row, col].axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_original.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_orig)

        # Untransformed only
        fig_untrans, axes_untrans = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
        if isinstance(axes_untrans, np.ndarray):
            axes_untrans = axes_untrans.reshape(n_rows, n_cols)
        else:
            axes_untrans = np.array([[axes_untrans]])
        for i in range(n_samples):
            row, col = i // n_cols, i % n_cols
            ax = axes_untrans[row, col]
            ax1.set_facecolor('white')
            color_opt = c_correct if pred_untrans[i] == target[i] else c_incorrect
            _plot_stroke_on_ax(x_untransformed[i].cpu(), ax, color=color_opt)
        for i in range(n_samples, n_rows * n_cols):
            row, col = i // n_cols, i % n_cols
            axes_untrans[row, col].axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_optimized.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_untrans)

        print(f"Saved: {save_prefix}_original.pdf and {save_prefix}_optimized.pdf")

def _plot_stroke_on_ax(seq, ax, linewidth=1.5, color="black"):
    """Plot single stroke sequence (N,4): dx,dy,pen,mask."""
    if isinstance(seq, torch.Tensor):
        seq = seq.detach().cpu().numpy()
    seq = np.asarray(seq)
    if seq.size == 0:
        return

    mask = seq[:, 3] if seq.shape[1] >= 4 else np.ones(seq.shape[0], dtype=bool)
    real_idx = mask == 1
    if not np.any(real_idx):
        return

    dx = seq[real_idx, 0]
    dy = seq[real_idx, 1]
    pen = seq[real_idx, 2] if seq.shape[1] >= 3 else np.zeros_like(dx)

    x = np.cumsum(dx)
    y = np.cumsum(dy)

    xs, ys = [], []
    for xi, yi, p in zip(x, y, pen):
        xs.append(xi)
        ys.append(-yi)
        if p == 1:
            ax.plot(xs, ys, linewidth=linewidth, color=color)
            xs, ys = [], []
    if xs:
        ax.plot(xs, ys, linewidth=linewidth, color=color)

    ax.axis("equal")
    ax.axis("off")

In [ ]:
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "a_visualization", dataset,
                           transform_name)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
from src.utils.eval.vis import plt_setup_latex


In [ ]:
if dataset == "bigger_mnist":
    from src.utils.eval.vis import plt_setup_latex
    import os
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.offsetbox import OffsetImage, AnnotationBbox

    # -----------------------------
    # Variable: number of images to show
    # -----------------------------
    num_images_to_show = 5

    # -----------------------------
    # 1. Setup data and parameters
    # -----------------------------
    model.eval()
    img, target = next(iter(test_loader))
    img = img[5:6].to(device)

    angles = torch.linspace(-np.pi, np.pi, 50).to(device)
    angles_np = angles.cpu().numpy()

    confidences = []
    rotated_images = []

    x = img.clone()

    # -----------------------------
    # 2. Compute confidence landscape
    # -----------------------------
    with torch.no_grad():
        dim_latent = best_problem.transform_sequence.initial_param(1).shape[-1]

        for angle in angles:
            params = torch.zeros((1, dim_latent), device=device)
            params[0, 0] = angle

            conf, _ = best_problem.calculate_error(x, params)
            conf = conf.item()

            x_rot = best_problem.transform_sequence.application_method(
                img,
                best_problem.transform_sequence(params)
            )

            confidences.append(conf)
            rotated_images.append(
                x_rot[0].detach().cpu().permute(1, 2, 0).numpy()
            )

    confidences = np.array(confidences)

    # -----------------------------
    # 3. Create inset figure
    # -----------------------------
    plt_setup_latex()

    fig, ax = plt.subplots(figsize=(1, 0.5)) # Slightly larger for clarity

    xmin, xmax = angles_np.min(), angles_np.max()
    ymin, ymax = confidences.min(), confidences.max()

    # --- INCREASED MARGINS ---
    # Increased margins ensure the plot doesn't touch the arrows (splines)
    xmargin = 0.3 * (xmax - xmin)
    ymargin = 0.22 * (ymax - ymin)

    ax.set_xlim(xmin - xmargin, xmax + xmargin)
    ax.set_ylim(ymin - ymargin, ymax + ymargin * 2.5) # Extra space at top for images

    # -----------------------------
    # 4. Plot solid curve BEHIND images
    # -----------------------------
    ax.plot(
        angles_np,
        confidences,
        color="blue",
        linewidth=1.0,
        alpha=1.0,
        zorder=1
    )

    # -----------------------------
    # 5. Plot images and markers
    # -----------------------------
    indices_to_show = np.linspace(
        0,
        len(rotated_images) - 1,
        num_images_to_show,
        dtype=int
    )

    zoom = 0.3
    y_offset_val = 0.1 * (ymax - ymin)

    for idx in indices_to_show:
        im = rotated_images[idx]
        im = (im - im.min()) / (im.max() - im.min() + 1e-8)


        ax.scatter(
            angles_np[idx],
            confidences[idx],
            color="blue",
            s=8,
            zorder=5,
            edgecolor='white',
            linewidth=0.3
        )

        imagebox = OffsetImage(
            im,
            zoom=zoom,
            cmap="gray" if im.shape[-1] == 1 else None
        )

        # Using AnnotationBbox with a connector line
        ab = AnnotationBbox(
            imagebox,
            (angles_np[idx], confidences[idx]),
            xybox=(0, 20),
            xycoords='data',
            boxcoords="offset points",
            frameon=False,
            pad=0,
            arrowprops=dict(arrowstyle="-", color="gray", alpha=0.5, lw=0.5),
            zorder=2
        )

        ax.add_artist(ab)

    # -----------------------------
    # 6. Plot transparent curve ON TOP
    # -----------------------------
    ax.plot(
        angles_np,
        confidences,
        color="blue",
        linewidth=1.0,
        alpha=0.35,
        zorder=3
    )

    # -----------------------------
    # 7. Remove default spines and ticks
    # -----------------------------
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_xticks([])
    ax.set_yticks([])

    # -----------------------------
    # 8. Draw arrow axes (with padding)
    # -----------------------------
    arrowprops = dict(
        arrowstyle="-|>",
        linewidth=0.6,
        color="black",
        shrinkA=0,
        shrinkB=0,
        mutation_scale=6
    )

    # Origin for the arrows (slightly offset from data)
    origin_x = xmin - 0.5 * xmargin
    origin_y = ymin - 0.5 * ymargin

    # x-axis arrow
    ax.annotate(
        "",
        xy=(xmax + 0.5 * xmargin, origin_y),
        xytext=(origin_x, origin_y),
        arrowprops=arrowprops,
        zorder=4
    )

    # y-axis arrow
    ax.annotate(
        "",
        xy=(origin_x, ymax + ymargin),
        xytext=(origin_x, origin_y),
        arrowprops=arrowprops,
        zorder=4
    )

    # -----------------------------
    # 9. Final layout and save
    # -----------------------------
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    plt.savefig(
        os.path.join(figure_path, "energy_landscape.pdf"),
        bbox_inches="tight",
        pad_inches=0.02
    )

    plt.show()

In [ ]:
#print label distribution of test set
import matplotlib.pyplot as plt
import numpy as np

# Extract labels from test set
dataset2 = test_loader.dataset
labels = []
for i in range(len(dataset2)):
    _, label = dataset2[i]
    labels.append(label)
labels = np.array(labels)

# Plot label distribution
plt.figure(figsize=(8, 4))
plt.hist(labels, bins=np.arange(labels.min(), labels.max() + 2) - 0.5, rwidth=0.8)
plt.xlabel("Class Label")
plt.ylabel("Count")
plt.title("Test Set Label Distribution")
plt.xticks(np.unique(labels))
plt.tight_layout()
plt.show()


In [ ]:
# Extract targets by iterating
dataset2 = test_loader.dataset
targets = []
for i in range(len(dataset2)):
    _, target = dataset2[i]
    targets.append(target)
targets = np.array(targets)

In [ ]:
dataset

In [ ]:
# Recreate test loaders with pre-shuffling to ensure class diversity
from torch.utils.data import DataLoader, Subset
import numpy as np

def create_class_balanced_subset(dataset, n_samples_per_class: int = 4):
    """
    Create a subset with n_samples_per_class from each class.
    Handles datasets that return 2 or 3 values.
    """
    # Get all targets
    if hasattr(dataset, 'targets'):
        targets = np.array(dataset.targets)
    elif hasattr(dataset, 'labels'):
        targets = np.array(dataset.labels)
    else:
        # Extract targets by iterating
        targets = []
        for i in range(len(dataset)):
            item = dataset[i]
            # Handle both (data, target) and (data, params, target)
            target = item[-1]  # Target is always the last element
            targets.append(target)
        targets = np.array(targets)

    # Get unique classes
    unique_classes = np.unique(targets)
    #order them
    unique_classes = np.sort(unique_classes)
    print(unique_classes)

    # Sample indices for each class
    #set generator for reproducibility
    np.random.seed(42)

    selected_indices = []
    for cls in unique_classes:
        cls_indices = np.where(targets == cls)[0]
        # Shuffle and select
        np.random.shuffle(cls_indices)
        selected = cls_indices[:n_samples_per_class]
        selected_indices.extend(selected)

    # Shuffle the final selection
    np.random.shuffle(selected_indices)

    return Subset(dataset, selected_indices)

# Create balanced subsets
n_samples_per_class = 4  # Adjust based on n_rows * n_cols / n_classes
test_subset = create_class_balanced_subset(test_loader.dataset, n_samples_per_class)
test_subset_transformed = create_class_balanced_subset(test_loader_transformed.dataset, n_samples_per_class)

# Create loaders
test_loader= DataLoader(
    test_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader_transformed = DataLoader(
    test_subset_transformed,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"Created balanced test loaders with {len(test_subset)} samples")
print(f"Expected classes: {n_classes}, samples per class: {n_samples_per_class}")


In [ ]:
# Visualize best detector
print("Best Detector Results:")
res_best = visualize_single_detector_transformations(
    model=model,
    optimizer=optimizer,
    problem=best_problem,
    test_loader=test_loader_transformed,
    dataset_info=dataset_info,
    n_rows=1,
    n_cols=1,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_best_detector_single")
)



In [ ]:
# Visualize best detector
print("Best Detector Results:")
res_best = visualize_single_detector_transformations(
    model=model,
    optimizer=optimizer,
    problem=best_problem,
    test_loader=test_loader_transformed,
    dataset_info=dataset_info,
    n_rows=2,
    n_cols=2,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_best_detector_2x2"),

)



In [ ]:
# Visualize best detector
print("Best Detector Results:")
res_best = visualize_single_detector_transformations(
    model=model,
    optimizer=optimizer,
    problem=best_problem,
    test_loader=test_loader_transformed,
    dataset_info=dataset_info,
    n_rows=2,
    n_cols=2,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_best_detector_2x2_original_sample"),
    second_loader=test_loader,
    second_indices = [0],
)




In [ ]:
# Visualize best detector
print("Best Detector Results:")
res_best = visualize_single_detector_transformations(
    model=model,
    optimizer=optimizer,
    problem=best_problem,
    test_loader=test_loader_transformed,
    dataset_info=dataset_info,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_best_detector")
)



In [ ]:
# Visualize best detector
print("ITS Detector Results:")
transform_seq_its = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)
its_problem = TransformationProblem(None, transform_seq_its, consolidate_method="consolidate_simple", max_batch_size=dataset_info.batch_size_search)

if dataset == "modelnet10":
    its_problem = ITSWRAPPER._prepare_problem(its_problem)



res_best = visualize_single_detector_transformations(
    model=model,
    optimizer=its_opt,
    problem=its_problem,
    test_loader=test_loader_transformed,
    dataset_info=dataset_info,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_its")
)



In [ ]:
# Visualize energy model
print("\nEnergy Model Results:")
res_energy = visualize_single_detector_transformations(
    model=model,
    optimizer=optimizer_240,
    problem=problem_energy_quarter,
    test_loader=test_loader_transformed,
    dataset_info=dataset_info,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_energy_model")
)

In [ ]:
label_map = None

In [ ]:
dataset

In [ ]:
from dataset.tu_berlin_sketch import cached_load_tu_berlin_data

if dataset=="tu_berlin":
    from typing import Dict

    def build_label_map_from_tu_berlin(data_dir: str, max_len: int = 200, interpolation_points: int = 2) -> Dict[int, str]:
        """
        Load TU﷿﷿﷿Berlin cached data (or regenerate) and return a mapping from numeric index -> class name.
        Example:
            build_label_map_from_tu_berlin('data') -> {0: 'airplane', 1: 'apple', ...}
        """
        # cached_load_tu_berlin_data is defined earlier in this module
        _, _, class_names = cached_load_tu_berlin_data(data_dir, max_len=max_len, interpolation_points=interpolation_points)
        return {i: name for i, name in enumerate(class_names)}
    label_map = build_label_map_from_tu_berlin(os.path.join(current_path, "experiment_files", "data"))


In [ ]:
if dataset=="modelnet10":
    label_map = {
        0: 'bathtub',
        1: 'bed',
        2: 'chair',
        3: 'desk',
        4: 'dresser',
        5: 'monitor',
        6: 'night stand',
        7: 'sofa',
        8: 'table',
        9: 'toilet'
    }

In [ ]:
def visualize_test_loader(
    test_loader,
    dataset_info,
    n_rows: int = 2,
    n_cols: int = 8,
    title: Optional[str] = None,
    save_path: Optional[str] = None
):
    """
    Visualize samples from test loader.
    Supports images, point clouds, and stroke sequences.

    Args:
        n_rows: Number of rows in the grid
        n_cols: Number of columns in the grid
        title: Optional title for the plot (shown in display only)
        save_path: If provided, saves the visualization as PDF (without title)
    """
    n_samples = n_rows * n_cols

    # Get data
    batch = next(iter(test_loader))
    data = batch[0][:n_samples]
    labels = batch[1][:n_samples] if len(batch) > 1 else None

    # Determine data type
    data_shape = data.shape

    if len(data_shape) == 4 and data_shape[1] in [1, 3]:  # Images (B, C, H, W)
        _visualize_images(data, labels, title, n_rows, n_cols, save_path)
    elif len(data_shape) == 3 and data_shape[-1] == 3:  # Point clouds (B, N, 3)
        _visualize_pointclouds(data, labels, title, n_rows, n_cols, save_path)
    elif len(data_shape) == 3 and data_shape[-1] == 4:  # Strokes (B, N, 4)
        _visualize_strokes(data, labels, title, n_rows, n_cols, save_path)
    else:
        print(f"Unsupported data shape: {data_shape}")


def _visualize_images(data, labels, title, n_rows, n_cols, save_path):
    """Helper to visualize image data."""
    grid = torchvision.utils.make_grid(data.cpu(), nrow=n_cols, normalize=True, padding=2)

    # Display with title
    if title:
        plt.figure(figsize=(12, 6 * n_rows / n_cols))
        plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray' if data.shape[1] == 1 else None)
        plt.title(title)
        plt.axis('off')
        plt.tight_layout()


    # Save without title
    if save_path:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6 * n_rows / n_cols))
        ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray' if data.shape[1] == 1 else None)
        ax.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(save_path)
        print(f"Saved: {save_path}")


    plt.show()

def _visualize_pointclouds(data, labels, title, n_rows, n_cols, save_path):
    """Helper to visualize point cloud data."""
    n_samples = min(len(data), n_rows * n_cols)

    fig = plt.figure(figsize=(W*4, n_rows *W/ n_cols*4))

    for i in range(n_samples):
        ax = fig.add_subplot(n_rows, n_cols, i + 1, projection='3d',rasterized=True)
        pts = data[i].cpu().numpy()
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        if labels is not None and label_map is not None:
            ax.set_title(f"{label_map[labels[i].item()]}", fontsize=32)

    if title:
        fig.suptitle(title)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()


def _visualize_strokes(data, labels, title, n_rows, n_cols, save_path):
    """Helper to visualize stroke sequence data."""
    n_samples = min(len(data), n_rows * n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(W*4,  n_rows*W*4 / n_cols ))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for i in range(n_samples):
        row, col = i // n_cols, i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]

        seq = data[i].cpu().numpy()
        mask = seq[:, 3] == 1
        if np.any(mask):
            dx, dy, pen = seq[mask, 0], seq[mask, 1], seq[mask, 2]
            x, y = np.cumsum(dx), np.cumsum(dy)

            xs, ys = [], []
            for xi, yi, p in zip(x, y, pen):
                xs.append(xi)
                ys.append(-yi)
                if p == 1:
                    ax.plot(xs, ys, 'k-', linewidth=1.5)
                    xs, ys = [], []
            if xs:
                ax.plot(xs, ys, 'k-', linewidth=1.5)
        ax.axis('equal')
        ax.axis('off')
        if labels is not None and label_map is not None:
            ax.set_title(f"{label_map[labels[i].item()]}", fontsize=28)

    # Hide extra subplots
    for i in range(n_samples, n_rows * n_cols):
        row, col = i // n_cols, i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.axis('off')

    if title:
        fig.suptitle(title)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

In [ ]:
 W = plt_setup_latex()

In [ ]:

# Save without title
visualize_test_loader(
    test_loader,
    dataset_info,
    n_rows=2,
    n_cols=8,
    save_path=os.path.join(figure_path, "test_loader_original.pdf")
)
plt.show()



In [ ]:
visualize_test_loader(
    test_loader_transformed,
    dataset_info,
    n_rows=2,
    n_cols=8,
    save_path=os.path.join(figure_path, "test_loader_transformed.pdf")
)
plt.show()


In [ ]:
import os
import pickle

datasets = ["bigger_mnist", "bigger_emnist", "coil100", "tu_berlin", "modelnet10"]

latex_table = [
    r"\begin{tabular}{l|c}",
    r"Dataset & Accuracy \\ \hline"
]

for ds in datasets:
    regression_cache_path = os.path.join(
        embedding_cache_path,
        f"regression_params_{ds}_{default_architecutre_mapping[ds]}_{get_dataset_info(ds).transform_seq_name}_test_set_60.pkl"
    )
    if not os.path.exists(regression_cache_path):
        print(f"Cache not found for {ds}, skipping.")
        continue
    with open(regression_cache_path, 'rb') as f:
        cache_data = pickle.load(f)
        test_acc_sim = cache_data.get('test_acc_sim', 0.0)
    latex_table.append(f"{ds} & {test_acc_sim:.2f} \\\\")

latex_table.append(r"\end{tabular}")

print("\n".join(latex_table))

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np

def visualize_regression_transformations(
    model,
    regression_dataloader,
    transform_seq_spatial,
    dataset_info,
    n_rows: int = 2,
    n_cols: int = 4,
    device: str = "cuda",
    save_prefix: Optional[str] = None
):
    """
    Visualize regression dataset transformations using stored parameters.
    Green border = correct classification, Red border = incorrect classification.

    Args:
        model: Classification model
        regression_dataloader: DataLoader with (data, params, target) tuples
        transform_seq_spatial: Transformation sequence to apply
        dataset_info: Dataset information
        n_rows: Number of rows in visualization grid
        n_cols: Number of columns in visualization grid
        device: Device to run on
        save_prefix: If provided, saves two PDFs: {prefix}_original.pdf and {prefix}_optimized.pdf
    """
    model.eval()
    n_samples = n_rows * n_cols

    # Get a batch from regression dataset
    data_batch = next(iter(regression_dataloader))
    data, params, target = data_batch
    data = data[:n_samples].to(device)
    params = params[:n_samples].to(device)
    target = target[:n_samples].to(device)

    # Apply stored transformation parameters to untransform
    T = transform_seq_spatial(params)
    x_untransformed = transform_seq_spatial.application_method(data, T)

    # Get predictions
    with torch.no_grad():
        pred_orig = model(data).argmax(dim=-1)
        pred_untrans = model(x_untransformed).argmax(dim=-1)

    # Determine data type
    data_shape = data.shape
    is_image = len(data_shape) == 4 and data_shape[1] in [1, 3]
    is_pointcloud = len(data_shape) == 3 and data_shape[-1] == 3
    is_stroke = len(data_shape) == 3 and data_shape[-1] == 4

    if is_image:
        _visualize_images_with_borders(
            data, x_untransformed, target, pred_orig, pred_untrans,
            n_rows, n_cols, save_prefix,
        )
    elif is_pointcloud:
        _visualize_pointclouds_with_borders(
            data, x_untransformed, target, pred_orig, pred_untrans,
            n_rows, n_cols, save_prefix,
        )
    elif is_stroke:
        _visualize_strokes_with_borders(
            data, x_untransformed, target, pred_orig, pred_untrans,
            n_rows, n_cols, save_prefix,
        )
    else:
        print(f"Unsupported data shape: {data_shape}")
        return

    # Print statistics
    acc_orig = (pred_orig == target).float().mean().item()
    acc_untrans = (pred_untrans == target).float().mean().item()

    print(f"Accuracy - Transformed: {acc_orig:.3f}, Untransformed (Canonical): {acc_untrans:.3f}")

    return {"accuracy_transformed": acc_orig, "accuracy_untransformed": acc_untrans}







In [ ]:
# Create regression loader with same balanced subset
regression_subset = create_class_balanced_subset(regression_dataset, n_samples_per_class)



regression_dataloader_balanced = DataLoader(
    regression_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"Created balanced regression loader with {len(regression_subset)} samples")

# Visualize regression transformations with balanced subset
print("\nRegression Dataset Canonical Transformation (Balanced):")
res_regression = visualize_regression_transformations(
    model=model,
    regression_dataloader=regression_dataloader_balanced,
    transform_seq_spatial=transform_seq_spatial,
    dataset_info=dataset_info,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=os.path.join(figure_path, "vis_regression_canonical")
)